In [ ]:
from pathlib import Path
import sys

candidate_roots = [Path.cwd(), *Path.cwd().parents]
for candidate_root in candidate_roots:
    if (candidate_root / "quacs" / "plumerise").exists():
        project_root = candidate_root
        break
else:
    raise RuntimeError("Run this notebook from the repository root or plumerise examples folder.")

repo_path = str(project_root)
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

from quacs.plumerise import compute_wildfire_profile_fraction_driver
from quacs.plumerise.layer_fraction_driver import layer_fraction_driver
from quacs.plumerise.prm_height_driver import prm_height_driver
from quacs.plumerise.pyrocb_flag_driver import pyrocb_flag_driver

print("Project root:", project_root)


## Using Synthetic sounding and fire inputs

In [ ]:
import numpy as np

np.set_printoptions(precision=6, suppress=True)

# Synthetic sounding and fire inputs.
z = np.arange(0.0, 16000.0, 1000.0)  # m AGL
t = np.where(z <= 10000.0, 300.0 - 6.5 * (z / 1000.0), 235.0)  # K
p = 1013.25 * np.exp(-z / 8000.0)  # hPa
u = np.ones_like(z)  # m s^-1
v = np.zeros_like(z)  # m s^-1
qv = np.full_like(z, 0.01)  # kg kg^-1

vegetation_class = "forest"  # unitless lookup key
fire_size_mean = 20_000_000.0  # m^2
fire_size_std = 0.0  # m^2

print("Synthetic sounding and fire inputs")
print("z[:6] m AGL:", z[:6])
print("p[:6] hPa:", p[:6])
print("t[:6] K:", t[:6])
print("u[:6] m/s:", u[:6])
print("v[:6] m/s:", v[:6])
print("qv[:6] kg/kg:", qv[:6])
print("vegetation_class:", vegetation_class)
print("fire_size_mean m^2:", fire_size_mean)


## Step 1: Diagnose PyroCb Flag

Input is direct sounding/fire variables. Output is only `pyrocb_flag`.

In [ ]:
pyrocb_flag = pyrocb_flag_driver(
    z,
    p,
    t,
    u,
    v,
    qv,
    vegetation_class,
    fire_size_mean,
    fire_size_std,
)

print("Step 1 output")
print("pyrocb_flag:", pyrocb_flag)
assert isinstance(pyrocb_flag, bool)


## Step 2: Resolve Injection Heights

Current implementation accepts Freitas-ready direct variables but returns fixed baseline heights plus optional pyroCb-adjusted heights.

In [ ]:
(
    injectH_base_m,
    injectH_top_m,
    injectH_pyroCb_base_m,
    injectH_pyroCb_top_m,
) = prm_height_driver(
    z,
    p,
    t,
    u,
    v,
    qv,
    vegetation_class,
    fire_size_mean,
    fire_size_std,
    pyrocb_flag,
)

print("Step 2 outputs")
print("injectH_base_m:", injectH_base_m)
print("injectH_top_m:", injectH_top_m)
print("injectH_pyroCb_base_m:", injectH_pyroCb_base_m)
print("injectH_pyroCb_top_m:", injectH_pyroCb_top_m)
assert injectH_top_m > injectH_base_m
if pyrocb_flag:
    assert injectH_pyroCb_base_m is not None
    assert injectH_pyroCb_top_m is not None


## Step 3: Build Layer Fraction

Input is direct height variables and `pyrocb_flag`. Output is only `layer_fraction`.

In [ ]:
layer_fraction_step3 = layer_fraction_driver(
    z,
    vegetation_class,
    injectH_base_m,
    injectH_top_m,
    injectH_pyroCb_base_m,
    injectH_pyroCb_top_m,
    pyrocb_flag,
)

print("Step 3 output")
print("layer_fraction:", layer_fraction_step3)
print("sum(layer_fraction):", layer_fraction_step3.sum())
print("nonzero layer fractions:")
for layer_index, fraction in enumerate(layer_fraction_step3):
    if fraction > 0.0:
        print(f"  k={layer_index:02d}, z_lower={z[layer_index]:7.1f} m, fraction={fraction:.6f}")

assert np.isclose(layer_fraction_step3.sum(), 1.0)
assert layer_fraction_step3.shape == z.shape


## Full Outer Driver

The public driver should produce the same `layer_fraction` as running Step 1, Step 2, and Step 3 explicitly.

In [ ]:
layer_fraction_outer = compute_wildfire_profile_fraction_driver(
    z,
    p,
    t,
    u,
    v,
    qv,
    vegetation_class,
    fire_size_mean,
    fire_size_std,
)

matches_step_run = np.allclose(layer_fraction_outer, layer_fraction_step3)

print("Outer-driver output")
print("layer_fraction:", layer_fraction_outer)
print("sum(layer_fraction):", layer_fraction_outer.sum())
print("outer driver matches explicit three-step run:", matches_step_run)

assert matches_step_run
assert np.isclose(layer_fraction_outer.sum(), 1.0)
